In [41]:
import sys
from pathlib import Path

# sobe um nível a partir de jobs/
sys.path.append(str(Path.cwd().parent))
from src.config.secrets import get_secret

# ------------------------------------------
# Azure Storage
# ------------------------------------------

storage_account = get_secret("AZURE-STORAGE-ACCOUNT", "AZURE_STORAGE_ACCOUNT")
storage_key = get_secret("AZURE-STORAGE-KEY", "AZURE_STORAGE_KEY")
container_name = get_secret("AZURE-CONTAINER-BRONZE", "AZURE_CONTAINER_BRONZE")

# ------------------------------------------
# Validações
# ------------------------------------------

if not storage_account:
    raise ValueError(
        "❌ AZURE-STORAGE-ACCOUNT não encontrada."
    )

if not storage_key:
    raise ValueError(
        "❌ AZURE-STORAGE-KEY não encontrada."
    )

if not container_name:
    raise ValueError(
        "❌ AZURE-CONTAINER-BRONZE não encontrada."
    )

# ------------------------------------------
# Cliente Azure Blob Storage
# ------------------------------------------

account_url = (f"https://{storage_account}.blob.core.windows.net")

blob_service_client = BlobServiceClient(account_url=account_url, credential=storage_key)

print("✅ Conexão com Azure Storage estabelecida")
print(f"✅ Container: {container_name}")


✅ Conexão com Azure Storage estabelecida
✅ Container: bronze


In [42]:
# Função que gera um registro fictício baseado na tabela INEP Alunos
def gerar_registro():
    return {
        "ano": random.choice([2023, 2024]),
        "id_municipio": str(random.randint(1000000, 9999999)),
        "id_escola": f"E{random.randint(1000,9999)}",
        "id_aluno": f"A{random.randint(100000,999999)}",
        "proficiencia": round(random.uniform(100, 300), 2),
        "data_ingestao": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }


In [43]:
# Gerar um lote de 100 registros simulados
df = pd.DataFrame([gerar_registro() for _ in range(100)])

# Visualizar os primeiros registros
print("👀 Visualização dos dados simulados:")
display(df.head())

# Converter para Parquet em memória
parquet_buffer = io.BytesIO()
df.to_parquet(parquet_buffer, index=False, engine="pyarrow")

# Nome do arquivo com partição por data
data_particao = datetime.datetime.now().strftime("%Y/%m/%d")
blob_name = f"inep_alunos_simulado/{data_particao}/dados.parquet"

# Upload para o container bronze
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)

print(f"✅ Dados simulados gravados no bronze em {blob_name}")


👀 Visualização dos dados simulados:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2024,1022377,E1547,A776373,171.02,2026-08-26 05:33:47
1,2023,8779650,E8572,A959990,283.73,2026-08-26 05:33:47
2,2023,4111621,E9855,A886180,247.98,2026-08-26 05:33:47
3,2023,6927964,E4952,A866873,242.13,2026-08-26 05:33:47
4,2024,1554520,E3311,A495104,142.07,2026-08-26 05:33:47


✅ Dados simulados gravados no bronze em inep_alunos_simulado/2026/08/26/dados.parquet


In [44]:
import time

# Simular streaming: gerar e gravar novos dados a cada 5 segundos
for i in range(5):  # número de ciclos
    df = pd.DataFrame([gerar_registro() for _ in range(10)])  # 10 registros por ciclo
    
    # Visualizar os primeiros registros de cada lote
    print(f"👀 Lote {i+1} - preview:")
    display(df.head())
    
    parquet_buffer = io.BytesIO()
    df.to_parquet(parquet_buffer, index=False, engine="pyarrow")
    
    data_particao = datetime.datetime.now().strftime("%Y/%m/%d/%H%M%S")
    blob_name = f"inep_alunos_streaming/{data_particao}/dados.parquet"
    
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
    blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)
    
    print(f"📤 Lote {i+1} enviado para {blob_name}")
    time.sleep(5)  # espera 5 segundos antes do próximo lote


👀 Lote 1 - preview:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,5324966,E6026,A129272,184.72,2026-08-26 05:33:48
1,2024,7251432,E7184,A931042,110.82,2026-08-26 05:33:48
2,2024,8510892,E7496,A543118,248.17,2026-08-26 05:33:48
3,2023,4014883,E8674,A852421,178.05,2026-08-26 05:33:48
4,2024,3715360,E7472,A659282,217.03,2026-08-26 05:33:48


📤 Lote 1 enviado para inep_alunos_streaming/2026/08/26/053348/dados.parquet
👀 Lote 2 - preview:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,6411978,E1488,A670464,209.04,2026-08-26 05:33:53
1,2023,8675488,E2579,A122431,236.50,2026-08-26 05:33:53
2,2024,2304419,E9215,A615248,191.75,2026-08-26 05:33:53
3,2024,3950722,E6360,A935851,288.89,2026-08-26 05:33:53
4,2024,8620808,E7166,A779708,161.65,2026-08-26 05:33:53


📤 Lote 2 enviado para inep_alunos_streaming/2026/08/26/053353/dados.parquet
👀 Lote 3 - preview:


...

📤 Lote 3 enviado para inep_alunos_streaming/2026/08/26/053358/dados.parquet
👀 Lote 4 - preview:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,2538205,E4556,A915967,100.07,2026-08-26 05:34:04
1,2024,8856395,E6097,A678939,246.66,2026-08-26 05:34:04
2,2024,8918154,E2976,A774353,192.60,2026-08-26 05:34:04
3,2024,7025743,E5726,A437638,124.31,2026-08-26 05:34:04
4,2023,8289014,E7991,A686916,281.05,2026-08-26 05:34:04


📤 Lote 4 enviado para inep_alunos_streaming/2026/08/26/053404/dados.parquet
👀 Lote 5 - preview:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,1908554,E4419,A145355,220.60,2026-08-26 05:34:11
1,2023,9738171,E8260,A870938,158.63,2026-08-26 05:34:11
2,2023,8346643,E1313,A336352,202.68,2026-08-26 05:34:11
3,2024,2415167,E4262,A612760,122.37,2026-08-26 05:34:11
4,2023,6782489,E2904,A606627,199.91,2026-08-26 05:34:11


📤 Lote 5 enviado para inep_alunos_streaming/2026/08/26/053411/dados.parquet
